In [ ]:
import pandas as pd

In [ ]:
def load_student_dataset(path):
    df = pd.read_csv(path)

    # Data standardization -> process of converting data from sources into a common, consistent format for the purpose of analysis.

    class_level_map = {
        10: "SS1",
        11: "SS2",
        12: "SS3"
    }

    df["English Language"] = pd.to_numeric(df["English Language"], errors="coerce")
    df["Literature in English"] = pd.to_numeric(df["Literature in English"], errors="coerce")

    #  Replaces all values. Any value for the column that's not mapped in the dictionary key, is replace with NaN (meaning an empty value)
    # df["class_level"] = df["class_level"].map(class_level_map)

    # Only replaces matched values. Does not attempt to replace values not provided in the dictionary key.
    df["class_level"] = df["class_level"].replace(class_level_map)

    return df

# SS1 Students with Over 92% Attendance Rate

In [ ]:
def filter_ss1_students_by_attendance_rate(df: pd.DataFrame, class_level = "SS1", attendance_rate = 92):
    """Returns a DataFrame of SS1 students with the minimum attendance rate, as provided"""
    results = df[
        (df["class_level"] == class_level) &
        (df["attendance"] > attendance_rate)
    ]

    return results[["student_id", "first_name", "last_name", "class_level", "attendance"]]

# Students with Over 85% in English Language and Literature in English

In [ ]:
def find_students_excelling_in_humanities(df, score = 85):
    results = df[
        (df["English Language"] > score) &
        (df["Literature in English"] > score)
    ]

    # results = df.query("'English Language' > @score and 'Literature in English' > @score")

    return results[["student_id", "first_name", "last_name", "English Language", "Literature in English"]]

In [ ]:
def find_students_excelling_in_humanities_alt(df, subjects = ["English Language", "Literature in English"], score = 85):
    queries = []

    for subject in subjects:
        queries.append(df[subject] > score)
    
    results = df[queries[0] & queries[1]]

    return results[["student_id", "first_name", "last_name", "English Language", "Literature in English"]]

In [ ]:
import random

# random.seed(80)
# TODO: Explain random seed

def find_students_needing_academic_support(df: pd.DataFrame, stem_subjects = None, max_score = 70):
    if stem_subjects is None:
        stem_subjects = ["Mathematics", "Chemistry", "Further Mathematics", "Physics", "Computer Science", "Biology"]


    # Approach 1
    # choice_subjects = random.choices(stem_subjects, k=2)

    # queries = []

    # for subject in choice_subjects:
    #     queries.append(df[subject] < max_score)

    # students_needing_support = df[
    #     (df["study_group"] == "Science") &
    #     queries[0] & queries[1]
    # ]

    # result_a = students_needing_support[["student_id", "first_name", "class_level", *choice_subjects]]
    # print(result_a)
    
    # print("\n")
    # print("=="*50)
    # print("\n")

    # Approach 2
    science_students = df[df["study_group"] == "Science"]
    students = [] # Needing support

    for index, student in science_students.iterrows():
        low_subjects = []

        for subject in stem_subjects:
            subject_score = student[subject]

            if subject_score < max_score:
                low_subjects.append(subject)

        if len(low_subjects) >= 2:
            student_info = {
                "students_id": student["student_id"],
                "first_name": student["first_name"],
                "last_name": student["last_name"],
                "low_subjects": ", ".join(low_subjects),
                "low_subjects_count": len(low_subjects),
            }

            students.append(student_info)

    result = pd.DataFrame(students)

    return result

In [ ]:
def find_potential_math_tutors(df: pd.DataFrame, class_level="SS3", min_score=90):
    """Returns a list of potential math tutors (students in SS3, with Mathematics scores above 90%)"""
    results = df[
        (df["class_level"] == class_level) &
        (df["Mathematics"] > min_score)
    ]

    return results[["student_id", "first_name", "last_name", "class_level", "Mathematics"]]

Sort the dataset by average score across all subjects they've taken (you'll need to calculate this first)

In [ ]:
def calc_average_score(df: pd.DataFrame):
    # You can either manually provide the list of columns or dynamically retrieve them as is done below

    # subject_columns = [
    #     "Agriculture", "Geography", "French", "Economics", "Biology", "Igbo", "History", "Physics", "Chemistry",
    #     "Further Mathematics", "Civic Education", "Computer Science", "Hausa", "English Language", "Government",
    #     "Mathematics", "Yoruba", "Literature in English"
    # ]

    df_copy = df.copy()

    columns_list = df_copy.columns.tolist()

    subject_columns = columns_list[-1:-19:-1][::-1]

    def calculate_student_average(row):
        scores = [row[subject] for subject in subject_columns if pd.notna(row[subject])]
        return round(sum(scores) / len(scores), 2)

    df_copy["average_score"] = df_copy.apply(calculate_student_average, axis=1)

    return df_copy.sort_values("average_score", ascending=False)[["student_id", "first_name", "last_name", "class_level", "average_score"]]

In [ ]:
students_df = load_student_dataset("../../data/students.csv")

#NOTE: Call the functions you want to test here!